## 損益計算書

In [9]:
import os
import sys
from dotenv import load_dotenv
from pathlib import Path

import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns

In [10]:
sys.path.append("/home/jovyan/notebook")
from Modules.get_market_data import GetMarketData
get_market_data = GetMarketData(Path('/home/jovyan/data'))

In [11]:
current_dir = Path.cwd()
data_path = (current_dir / "data").resolve()

#### [2]損益計算書（PL）のウォーターフォールチャート

In [12]:
# finance/data/7203_120/XBRL_TO_CSV/jpcrp030000-asr-001_E02144-000_2024-03-31_01_2024-06-25.csv
ufo_edi = pd.read_csv(
    data_path / "7203_120/XBRL_TO_CSV/jpcrp030000-asr-001_E02144-000_2024-03-31_01_2024-06-25.csv",
    encoding="utf-16",
    sep="\t",
    parse_dates=True
)
cond_pl = ufo_edi["相対年度"] == "当期"
cond_pl &= ufo_edi["連結・個別"] == "個別"
cond_pl &= ufo_edi["コンテキストID"] == "CurrentYearDuration_NonConsolidatedMember"
pl_df = ufo_edi.loc[cond_pl, ["要素ID", "項目名", "単位", "値"]].drop_duplicates()
pl_df.iloc[10:19, :]

,要素ID,項目名,単位,値
1529,jppfs_cor:OtherNOE,その他、営業外費用,円,115653000000
1531,jppfs_cor:NonOperatingExpenses,営業外費用,円,148447000000
1533,jppfs_cor:OrdinaryIncome,経常利益又は経常損失（△）,円,5578695000000
1535,jppfs_cor:IncomeBeforeIncomeTaxes,税引前当期純利益又は税引前当期純損失（△）,円,5578695000000
1537,jppfs_cor:IncomeTaxesCurrent,法人税、住民税及び事業税,円,1253728000000
1539,jppfs_cor:IncomeTaxesDeferred,法人税等調整額,円,-74888000000
1541,jppfs_cor:IncomeTaxes,法人税等,円,1178840000000
1543,jppfs_cor:ProfitLoss,当期純利益又は当期純損失（△）（平成26年3月28日財規等改正後）,円,4399855000000
1656,jppfs_cor:ReversalOfReserveForSpecialDepreciation,特別償却準備金の取崩,円,－


In [13]:
pl_df = pl_df.assign(**{"item": pl_df["項目名"].str.replace("（.*", "", regex=True)})
pl_df = pl_df.assign(**{"値": pl_df["値"].replace("－", "")})
pl_df = pl_df.assign(**{"value": pl_df["値"].replace("", 0).astype(int)})
pl_data = [
    ["売上高", "relative", 1],
    ["売上原価", "relative", -1],
    ["売上総利益又は売上総損失", "total", 1],
    ["販売費及び一般管理費", "relative", -1],
    ["営業利益又は営業損失", "total", 1],
    ["営業外収益", "relative", 1],
    ["営業外費用", "relative", -1],
    ["経常利益又は経常損失", "total", 1],
    ["法人税等", "relative", -1],
    ["当期純利益又は当期純損失", "total", 1],
]
statements, measures, signs = zip(*pl_data)
waterfall_data = pd.DataFrame(pl_data, columns=["statement", "measure", "sign"])
# 各PL項目に表示する金額テキストは、
# 各項目名で抽出した'value'は1000000で除算して100万円単位にする
waterfall_data = waterfall_data.assign(**{"value": waterfall_data["statement"].apply(
        lambda label: pl_df.set_index("item").loc[label, "value"] // 1000000
    ) * waterfall_data["sign"]
})
waterfall_data

,statement,measure,sign,value
0,売上高,relative,1,17575593
1,売上原価,relative,-1,-12919592
2,売上総利益又は売上総損失,total,1,4656000
3,販売費及び一般管理費,relative,-1,-1561506
4,営業利益又は営業損失,total,1,3094495
5,営業外収益,relative,1,2632647
6,営業外費用,relative,-1,-148447
7,経常利益又は経常損失,total,1,5578695
8,法人税等,relative,-1,-1178840
9,当期純利益又は当期純損失,total,1,4399855


In [14]:
waterfall_fig = go.Figure(go.Waterfall(
    x=statements,
    y=waterfall_data["value"],
    measure=measures,
    text=waterfall_data["value"].astype(str),
    textposition="outside",
))
# レイアウトでy軸の範囲を0円から売り上げより少し大きい金額までに設定
waterfall_fig.update_layout(width=900, height=450,
    yaxis = {
        "tickformat": ",.0f",
        "tickprefix": "百万円",
        "title": None,
        "range": [
            0,
            int(waterfall_data.set_index("statement").at["売上高", "value"]) 
            * 1.15,
        ],                      
    },
)
waterfall_fig.show()

### [3]貸借対照表(BS)

In [15]:
cond_bs = ufo_edi["項目名"].isin(["固定資産", "流動資産", "純資産", "固定負債", "流動負債"])
cond_bs &= ufo_edi["相対年度"].isin(["前期末", "当期末"])
cond_bs &= ufo_edi["連結・個別"] == "個別"
cond_bs &= ufo_edi["コンテキストID"].isin(["CurrentYearInstant_NonConsolidatedMember",
                                        "Prior1YearInstant_NonConsolidatedMember"])
balance = ufo_edi.loc[cond_bs].drop_duplicates()
# 金額を百万単位に変換
balance = balance.assign(**{"value": balance["値"].replace("－", "").replace("", "0").astype(int) // 1000000})

In [16]:
# 相対年度、項目名、値の列に絞ったテーブルを作成
bs_df = balance.pivot(index="相対年度", columns="項目名", values="value")
# 前期と当期の差額
diff_val = (bs_df.loc["当期末", "流動資産"] + bs_df.loc["当期末", "固定資産"]) - (
    bs_df.loc["前期末", "流動資産"] + bs_df.loc["前期末", "固定資産"]
)
# グラフ設定のための定数を用意
bar_width = 0.4
xpos = { "前期末": 0, "当期末": 1 }
cols_bs = ["固定資産", "流動資産", "純資産", "固定負債", "流動負債"]
# 資産が青系、負債が赤系、純資産が緑になるように色を設定
bs_colors = {
    "固定資産": "#1f77b4",
    "流動資産": "#aec7e8",
    "純資産": "#2ca02c",
    "固定負債": "#d62728",
    "流動負債": "#ff9896"
}
# 純資産（左）と負債・純資産（右）に分ける
offsetgrps = { "固定資産": 0, "流動資産": 0, "純資産": 1, "固定負債": 1, "流動負債": 1 }
# 積上棒グラフの開始位置に使用
base_dict = {
    "流動資産": bs_df["流動資産"],
    "固定負債": bs_df["純資産"],
    "流動負債": bs_df["純資産"] + bs_df["固定負債"],
}
# グラフ描画領域の作成
bs_fig = go.Figure()
# 各棒グラフを追加
for col in cols_bs:
    base = base_dict[col] if col in base_dict else None
    bs_fig.add_trace(go.Bar(
        x=[xpos[i] for i in bs_df.index],
        y=bs_df[col], 
        marker_color=bs_colors[col],
        offsetgroup=offsetgrps[col],
        base=base,
        width=bar_width,
        text=[f"{col}:{val} 百万円" for val in bs_df[col]],
        textposition="inside",
        showlegend=False,
    ))
# 前期と当期の変化を明示する線を追加
bs_fig.add_trace(go.Scatter(
    x=[0 + bar_width, 1 - bar_width],
    y=bs_df["流動資産"] + bs_df["固定資産"],
    mode="lines",
    showlegend=False,
))
# レイアウトの更新
bs_fig.update_layout(width=1000, height=500,
    barmode="group",
    xaxis = { "tickvals": [0, 1], "ticktext": ["前期末", "当期末"] },
    yaxis = { "tickformat": ",.0f", "tickprefix": "百万円" },
    # 前期と当期の差額の金額をアノテーションを用いて表示
    annotations=[
        {
            "x": (xpos["前期末"] + xpos["当期末"]) / 2,
            "y": (bs_df.loc["当期末", "流動資産"] + bs_df.loc["当期末", "固定資産"]) * 1.1,
            "text": str(diff_val) + " 百万円",
            "font": { "size": 15 },
            "showarrow": False,
        }
    ],
)
bs_fig.show()
